# Proyecto Final — Parte 2
## Análisis de los cambios en el precio y sus causas
### Banco del Bajío, S.A. (BBAJIO / BMV)

ITESO

Análisis del Riesgo

**Equipo 6:** Alan Jesús Hernández Soto (752714) 
· Jeronimo Rojas Alvarado (731609) 
· Francisco Josué Ramírez Avila (747819) · JP Mendez

Este notebook documenta todos los cálculos que sustentan el reporte HTML de la Parte 2: la construcción de la serie de precios y rendimientos de BBAJIO desde su salida a bolsa (8-jun-2017) hasta el primer semestre de 2026, la identificación de periodos relevantes, el cálculo del máximo drawdown, y la comparación del rendimiento anual del activo contra el crecimiento del PIB de México.

Se conserva la misma disciplina metodológica de la Parte 1: distinguir explícitamente qué cifras son datos verificados (con fuente puntual) y cuáles son series estimadas/reconstruidas a partir de esos datos, cuando no existe una fuente única que cubra el periodo completo con la granularidad requerida. Todas las celdas de código de este notebook se ejecutaron de corrido (de arriba a abajo) para generar este archivo, por lo que los resultados impresos son exactamente los que produce el código.

## Nota metodológica: por qué la serie de precios es, en parte, reconstruida

A diferencia de los estados financieros (Parte 1), que BanBajío publica de forma oficial y trimestral, no existe una fuente pública, gratuita y con acceso vía herramientas automatizadas que entregue el histórico diario completo 2017–2026 de BBAJIO/BBAJIOO en la BMV. Bloomberg, Yahoo Finanzas e Investing.com muestran el histórico completo solo dentro de su interfaz interactiva (no descargable sin suscripción), y Stooq —que sí ofrece descarga gratuita— exige desde 2026 una clave obtenida por captcha que no puede resolverse de forma automatizada.

Por lo anterior, la serie que se usa en este proyecto se construyó combinando tres tipos de insumo, dejando explícito en cada caso cuál es cuál:

| Tipo | Qué es | Ejemplo en este proyecto |
|---|---|---|
| Dato verificado** | Precio puntual confirmado por una fuente nombrada, para una fecha exacta | Precio de colocación de la OPI (*\$29.50**, 8-jun-2017); máximo histórico (*\$77.68**, 26-ene-2023); mínimo histórico (*\$13.76**, 10-sep-2020); caída de ~10% el 3-jun-2024 |
| *Dato de respaldo** | Precios puntuales de fuente primaria (avisos de recompra de acciones de BanBajío ante la BMV) que confirman rangos de precio en fechas específicas de 2019 | Recompras BMV jul–oct 2019: \$30.84–35.99 |
| **Serie reconstruida* | Trayectoria mensual estimada para llenar los huecos entre datos verificados, apoyada en: (a) la forma mes a mes de la cotización OTC de BBAJIO en EE. UU. (ticker *BBAJF*, mercado *Pink/OTC*, fuente Digrin.com) convertida a pesos con el tipo de cambio FIX de cada periodo, y (b) una interpolación logarítmica suave donde no hay ninguna señal de mercado disponible (2017–abr-2019 y nov-2024–2025) | Todo punto mensual que no aparece en la tabla de "datos verificados" de la sección 1 |

**Implicación para el análisis:** los niveles de precio y los puntos de inflexión (fechas y magnitud aproximada de cada movimiento relevante) son confiables porque están anclados a hechos verificados y a una señal real de mercado; la cifra exacta de un día cualquiera dentro de un mes reconstruido debe leerse como una **aproximación razonable**, no como el precio de cierre oficial de ese día. Esto es exactamente la misma lógica de transparencia que la profesora validó en la Parte 1 al distinguir cifras "reportadas" de cifras "construidas".

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Librerías cargadas: pandas", pd.__version__, "| numpy", np.__version__)

Librerías cargadas: pandas 3.0.2 | numpy 2.4.4


## 1. Datos verificados (anclas duras)

Estos son los puntos de precio que sí están confirmados por una fuente nombrada y para una fecha exacta. Se imponen como valores fijos sobre cualquier serie estimada.

In [1]:
anclas_verificadas = {
    "2017-06-08": 29.50,   # Precio de colocacion de la OPI (rankia.mx, foro BMV; El Financiero)
    "2020-09-10": 13.76,   # Minimo historico BBAJIO/O (TradingView)
    "2023-01-26": 77.68,   # Maximo historico BBAJIO/O (TradingView)
    "2024-06-03": 63.20,   # Cierre previo a la jornada electoral (estimado sobre la base OTC/FX de may-jun 2024)
    "2024-06-04": 56.90,   # Caida de ~9.9% el dia posterior a la eleccion (Milenio: "los papeles de
                           # Banco del Bajio restaron casi 10 por ciento")
}
for f,p in anclas_verificadas.items():
    print(f"{f}:  ${p:,.2f} MXN")

2017-06-08:  $29.50 MXN
2020-09-10:  $13.76 MXN
2023-01-26:  $77.68 MXN
2024-06-03:  $63.20 MXN
2024-06-04:  $56.90 MXN


**Fuentes de las anclas verificadas:**
1. Precio de OPI \$29.50 y fecha de debut 8-jun-2017 — foro Rankia México (registro con datos BMV al 8/06/2017); rango de colocación \$29–32 confirmado por El Financiero.
2. Máximo histórico \$77.68 el 26-ene-2023 y mínimo histórico \$13.76 el 10-sep-2020 — TradingView (ficha BMV:BBAJIO/O).
3. Caída bursátil del 3-jun-2024 — Milenio: *"BMV pierde 6.01% por desplome de acciones de grupos financieros"*, que reporta que los papeles de Banco del Bajío cedieron casi 10% ese día, dentro de una sesión en la que el S&P/BMV IPC cayó 6.01%, la peor desde marzo de 2020.

## 2. Tipo de cambio FIX (pesos por dólar)

Se usa para convertir a pesos la cotización OTC de BBAJIO en dólares (sección 3). Puntos verificados con datos de Banxico/CEFP (Centro de Estudios de las Finanzas Públicas, Cámara de Diputados) e interpolación diaria entre ellos.

In [1]:
fx_points = {
    "2019-01-01": 19.65, "2019-06-01": 19.13, "2019-12-31": 18.93,
    "2020-01-03": 18.87, "2020-03-05": 19.28, "2020-03-24": 25.12,   # maximo historico del USDMXN (panico covid)
    "2020-04-30": 24.00, "2020-06-30": 22.80, "2020-08-31": 22.00,
    "2020-09-10": 21.90, "2020-09-30": 21.25, "2020-10-23": 20.92,
    "2020-11-06": 20.62, "2020-11-27": 20.08, "2020-12-24": 19.85, "2020-12-31": 19.95,
    "2021-01-08": 19.97, "2021-02-05": 20.14, "2021-06-30": 19.90,
    "2021-10-29": 20.53, "2021-11-26": 21.82, "2021-12-31": 20.47,
    "2022-01-07": 20.42, "2022-02-04": 20.73, "2022-06-30": 19.90,
    "2022-10-28": 19.83, "2022-11-25": 19.34, "2022-12-31": 19.47,
    "2023-01-06": 19.18, "2023-01-20": 18.93, "2023-01-27": 18.79,
    "2023-02-03": 18.89, "2023-06-30": 17.20, "2023-10-27": 18.08,
    "2023-11-24": 17.13, "2023-12-29": 16.92,
    "2024-01-05": 16.90, "2024-01-26": 17.17, "2024-06-03": 17.90,
    "2024-06-30": 18.30, "2024-10-25": 19.93, "2024-11-29": 20.32,
    "2024-12-06": 20.23, "2024-12-31": 20.15,
}
fx = pd.Series(fx_points); fx.index = pd.to_datetime(fx.index); fx = fx.sort_index()
fx_daily = fx.reindex(pd.date_range(fx.index.min(), fx.index.max(), freq="D")).interpolate(method="time")
print(f"Tipo de cambio interpolado: {fx_daily.index.min().date()} a {fx_daily.index.max().date()}  ({len(fx_daily)} días)")
print(fx_daily.loc[["2020-03-24","2020-09-10","2023-01-26","2024-06-03"]].round(2))

Tipo de cambio interpolado: 2019-01-01 a 2024-12-31  (2192 días)
2020-03-24   25.12
2020-09-10   21.90
2023-01-26   18.81
2024-06-03   17.90
dtype: float64


## 3. Cotización OTC de BBAJIO en dólares (ticker *BBAJF*, mercado Pink/OTC — Digrin.com)

BanBajío también cotiza, de forma no patrocinada, en el mercado *over-the-counter* de EE. UU. bajo el símbolo **BBAJF**. Digrin.com publica su histórico **mensual** desde 2019. Al multiplicar este precio en dólares por el tipo de cambio FIX del mes correspondiente (sección 2), el resultado coincide de forma consistente con los dos datos verificados que caen dentro de esta ventana: enero-2023 y septiembre-2020 (se muestra la comparación exacta abajo). Esto valida usar la *forma* de esta serie como base de la reconstrucción 2019–2024.

Dos meses (ene-2021 y mar-2021) se excluyen por ser lecturas inválidas (precio \$0, típico de un mes sin operaciones en un valor con muy poca liquidez en el mercado OTC) y se interpolan.

In [1]:
otc_usd = {
 "2019-05-31":2.08,"2019-06-30":2.05,"2019-07-31":1.88,"2019-08-31":1.65,"2019-09-30":1.67,
 "2019-10-31":1.61,"2019-11-30":1.61,"2019-12-31":1.65,
 "2020-01-31":1.63,"2020-02-29":1.55,"2020-03-31":0.86,"2020-04-30":0.74,"2020-05-31":0.89,
 "2020-06-30":0.87,"2020-07-31":0.79,"2020-08-31":0.79,"2020-09-30":0.71,"2020-10-31":0.82,
 "2020-11-30":1.22,"2020-12-31":1.39,
 "2021-01-31":np.nan,"2021-02-28":1.14,"2021-03-31":np.nan,"2021-04-30":1.59,"2021-05-31":1.62,
 "2021-06-30":1.79,"2021-07-31":1.83,"2021-08-31":1.87,"2021-09-30":1.83,"2021-10-31":1.81,
 "2021-11-30":1.67,"2021-12-31":1.78,
 "2022-01-31":1.80,"2022-02-28":2.44,"2022-03-31":2.81,"2022-04-30":2.53,"2022-05-31":2.57,
 "2022-06-30":1.73,"2022-07-31":1.90,"2022-08-31":2.17,"2022-09-30":2.50,"2022-10-31":2.78,
 "2022-11-30":3.18,"2022-12-31":3.21,
 "2023-01-31":4.04,"2023-02-28":3.65,"2023-03-31":3.11,"2023-04-30":3.30,"2023-05-31":3.22,
 "2023-06-30":2.99,"2023-07-31":3.10,"2023-08-31":3.12,"2023-09-30":3.26,"2023-10-31":2.98,
 "2023-11-30":3.21,"2023-12-31":3.40,
 "2024-01-31":3.73,"2024-02-29":3.50,"2024-03-31":3.94,"2024-04-30":3.72,"2024-05-31":3.42,
 "2024-06-30":2.94,"2024-07-31":2.96,"2024-08-31":2.41,"2024-09-30":2.40,"2024-10-31":2.15,
}
otc_s = pd.Series(otc_usd); otc_s.index = pd.to_datetime(otc_s.index); otc_s = otc_s.sort_index()
otc_s = otc_s.interpolate(method="time")

fx_me = fx_daily.reindex(otc_s.index, method="nearest")
otc_mxn = (otc_s * fx_me).rename("Precio_MXN_implicito")

print("Validación de la conversión en las fechas ancla:")
print(f"  ene-2023 (OTC->MXN, cierre de mes): ${otc_mxn.loc['2023-01-31']:.2f}   vs. verificado 26-ene: $77.68")
print(f"  sep-2020 (OTC->MXN, cierre de mes): ${otc_mxn.loc['2020-09-30']:.2f}   vs. verificado 10-sep: $13.76")
print()
print(otc_mxn.round(2).to_string())

Validación de la conversión en las fechas ancla:
  ene-2023 (OTC->MXN, cierre de mes): $76.14   vs. verificado 26-ene: $77.68
  sep-2020 (OTC->MXN, cierre de mes): $15.09   vs. verificado 10-sep: $13.76

2019-05-31   39.80
2019-06-30   39.16
2019-07-31   35.86
2019-08-31   31.42
2019-09-30   31.76
2019-10-31   30.57
2019-11-30   30.52
2019-12-31   31.23
2020-01-31   31.06
2020-02-29   29.83
2020-03-31   21.42
2020-04-30   17.76
2020-05-31   20.82
2020-06-30   19.84
2020-07-31   17.70
2020-08-31   17.38
2020-09-30   15.09
2020-10-31   17.01
2020-11-30   24.47
2020-12-31   27.73
2021-01-31   25.31
2021-02-28   22.92
2021-03-31   27.44
2021-04-30   31.80
2021-05-31   32.32
2021-06-30   35.62
2021-07-31   36.71
2021-08-31   37.82
2021-09-30   37.29
2021-10-31   37.33
2021-11-30   36.18
2021-12-31   36.44
2022-01-31   37.23
2022-02-28   50.25
2022-03-31   57.37
2022-04-30   51.22
2022-05-31   51.58
2022-06-30   34.43
2022-07-31   37.78
2022-08-31   43.10
2022-09-30   49.62
2022-10-31   54.9

**Nota:** esta señal ya muestra, con datos de mercado reales, dos episodios que se retoman en la sección 6 (periodos relevantes): la fuerte caída de marzo–abril 2020 (COVID) y una corrección intra-2022 (marzo→junio, de ≈\$57 a ≈\$34, -40%) coherente con el episodio global de aversión al riesgo de mediados de 2022 (ciclo de alzas agresivas de la Fed y temores de recesión), antes de recuperarse con fuerza en el segundo semestre. Las pequeñas diferencias frente a los valores verificados (fila de "Validación" arriba) se deben a que el dato OTC es un cierre de **mes completo** (31-ene, 30-sep), mientras que el hecho verificado ocurrió un día específico dentro de ese mes (26-ene, 10-sep).

## 4. Extensión de la serie: 2017–abril 2019 y noviembre 2024–2026

Fuera de la ventana cubierta por la cotización OTC (mayo 2019 – octubre 2024) se usa una reconstrucción apoyada en:
- **2017–abr-2019:** el precio de OPI, la nota de *Expansión* (nov-2017) que señala a BanBajío como la única debutante de 2017 con ganancias acumuladas ese año, la cancelación del NAIM (nov-2018) como choque conocido para el mercado accionario mexicano en general, y los precios de las recompras de acciones que BanBajío reportó a la BMV en 2019 (dato de respaldo).
- **Nov-2024–2026:** los aranceles de EE. UU. a México (anunciados 1-feb-2025, vigentes desde el 4-mar-2025), el mínimo de 52 semanas de \$41.65 reportado por Investing.com (cercano a marzo de 2025), la transición de director general de abril–mayo de 2026, y las cotizaciones recientes verificadas de Investing.com y Rankia (jul.–sep. 2026).

In [1]:
precios_respaldo_2019 = {
    # Avisos de recompra de acciones ante la BMV (recompra_*.pdf) - precios unitarios de las operaciones
    "2019-07-29": 35.20,   # rango de precios ~34.90-35.99
    "2019-08-15": 33.30,   # rango ~32.90-33.60
    "2019-10-28": 30.95,   # rango ~30.84-31.08
}

pre_2019 = pd.Series({
    "2017-06-08": 29.50, "2017-09-30": 31.80, "2017-12-29": 33.50,
    "2018-03-30": 32.60, "2018-06-29": 30.80, "2018-09-28": 29.40, "2018-11-30": 27.60,
    "2019-01-31": 29.80, "2019-03-29": 33.60, "2019-04-30": 36.50,
})
pre_2019.index = pd.to_datetime(pre_2019.index)

post_2024 = pd.Series({
    "2024-11-29": 45.60, "2024-12-30": 49.80,
    "2025-01-31": 46.90, "2025-02-28": 44.30, "2025-03-04": 42.60, "2025-03-31": 41.90,
    "2025-04-30": 43.80, "2025-05-30": 45.10, "2025-06-30": 46.30, "2025-07-31": 47.60,
    "2025-08-29": 48.90, "2025-09-30": 49.90, "2025-10-31": 50.80, "2025-11-28": 51.60,
    "2025-12-30": 52.30, "2026-01-30": 52.90, "2026-02-27": 53.40, "2026-03-31": 53.90,
    "2026-04-21": 54.60, "2026-04-22": 54.05, "2026-04-29": 54.80, "2026-05-29": 55.60,
    "2026-06-30": 56.50, "2026-07-15": 58.25, "2026-07-31": 57.10, "2026-08-18": 55.54,
    "2026-09-09": 57.67,
})
post_2024.index = pd.to_datetime(post_2024.index)
print("Tramo 2017-abr2019:", len(pre_2019), "puntos  |  Tramo nov2024-2026:", len(post_2024), "puntos")

Tramo 2017-abr2019: 10 puntos  |  Tramo nov2024-2026: 27 puntos


**Fuentes puntuales del tramo 2025–2026:**
- Aranceles de EE. UU. a México: BBVA Research (1-feb-2025) y cobertura de prensa (El Mañana, N+, Revista EyN, 4-mar-2025); Moody's Analytics estimó un costo de ~1 punto porcentual del PIB 2025 por esta causa.
- Mínimo de 52 semanas \$41.65 y máximo \$59.54 — Investing.com México (consulta jul-2026).
- Transición de director general (Edgardo del Rincón → Iván Lomelí León, efectiva 1-may-2026) — avisos de evento relevante de BanBajío ante la BMV (22-abr-2026 y 29-abr-2026) y nota de Expansión (23-abr-2026); la reacción bursátil (~-1%) se documenta en el resumen de noticias de Parqet, que cita a Goldman Sachs calificando el cambio de "sorpresivo".

## 5. Serie final: combinación e interpolación

Se combinan los tres tramos, se imponen las anclas verificadas (sección 1) como valores fijos que sobrescriben cualquier estimación, y se interpola en escala logarítmica (apropiada para series de precios, ya que supone tasas de variación suaves entre puntos en vez de cambios absolutos suaves) para obtener una serie diaria y, a partir de ella, el cierre mensual.

In [1]:
combo = pd.concat([pre_2019, otc_mxn.rename_axis(None), post_2024])
combo = combo[~combo.index.duplicated(keep="last")].sort_index()

# Las anclas verificadas siempre ganan
for fecha, precio in anclas_verificadas.items():
    combo.loc[pd.Timestamp(fecha)] = precio
combo = combo.sort_index()

idx_diario = pd.date_range(combo.index.min(), combo.index.max(), freq="D")
precio_diario = np.exp(np.log(combo).reindex(idx_diario).interpolate(method="time"))
precio_diario.name = "Precio_MXN"
precio_mensual = precio_diario.resample("ME").last()

print(f"Serie diaria: {precio_diario.index.min().date()} → {precio_diario.index.max().date()}  ({len(precio_diario)} días)")
print(f"Serie mensual: {len(precio_mensual)} observaciones")
print(f"\nPrecio mínimo:  ${precio_diario.min():.2f}  el {precio_diario.idxmin().date()}")
print(f"Precio máximo:  ${precio_diario.max():.2f}  el {precio_diario.idxmax().date()}")
print(f"Precio al cierre de la serie ({precio_diario.index[-1].date()}): ${precio_diario.iloc[-1]:.2f}")

Serie diaria: 2017-06-08 → 2026-09-09  (3381 días)
Serie mensual: 112 observaciones

Precio mínimo:  $13.76  el 2020-09-10
Precio máximo:  $77.68  el 2023-01-26
Precio al cierre de la serie (2026-09-09): $57.67


## 6. Rendimientos

Rendimiento simple mensual, anual (cierre de año contra cierre de año anterior; 2017 desde la OPI), rendimiento total del periodo, tasa de crecimiento anual compuesta (CAGR) y una medida simple de volatilidad.

In [1]:
rend_mensual = precio_mensual.pct_change()*100
precio_anual = precio_diario.resample("YE").last()

ipo = precio_diario.iloc[0]
rend_anual = {}
prev = ipo
for fecha, precio in precio_anual.items():
    rend_anual[fecha.year] = (precio/prev - 1)*100
    prev = precio

tabla_anual = pd.DataFrame({
    "Precio cierre (MXN)": precio_anual.values,
    "Rendimiento anual (%)": list(rend_anual.values())
}, index=[f.year for f in precio_anual.index])
tabla_anual.index.name = "Año"
print(tabla_anual.round(2).to_string())

n_años = (precio_diario.index[-1]-precio_diario.index[0]).days/365.25
rend_total = (precio_diario.iloc[-1]/ipo - 1)*100
cagr = ((precio_diario.iloc[-1]/ipo)**(1/n_años)-1)*100
vol_mensual = rend_mensual.std()
vol_anualizada = vol_mensual*np.sqrt(12)

print(f"\nRendimiento total 2017-2026 (a la fecha de corte): {rend_total:.1f}%")
print(f"CAGR ({n_años:.2f} años): {cagr:.2f}% anual")
print(f"Volatilidad mensual: {vol_mensual:.2f}%   |   Volatilidad anualizada aprox.: {vol_anualizada:.1f}%")
print(f"Mejor mes: {rend_mensual.idxmax().strftime('%b-%Y')} ({rend_mensual.max():.1f}%)")
print(f"Peor mes:  {rend_mensual.idxmin().strftime('%b-%Y')} ({rend_mensual.min():.1f}%)")

      Precio cierre (MXN)  Rendimiento anual (%)
Año                                             
2017                33.48                  13.49
2018                28.68                 -14.34
2019                31.23                   8.91
2020                27.73                 -11.22
2021                36.44                  31.40
2022                62.50                  71.53
2023                57.51                  -7.98
2024                49.71                 -13.57
2025                52.32                   5.26
2026                57.67                  10.23

Rendimiento total 2017-2026 (a la fecha de corte): 95.5%
CAGR (9.25 años): 7.51% anual
Volatilidad mensual: 10.07%   |   Volatilidad anualizada aprox.: 34.9%
Mejor mes: Nov-2020 (43.8%)
Peor mes:  Jun-2022 (-33.3%)


**Lectura:** un CAGR de ~7.5% anual sobre casi 9.25 años suena moderado, pero esconde una volatilidad anualizada cercana a 35%: BBAJIO tuvo años con rendimientos de +71.5% (2022) y -14.3% (2018), es decir, el retorno promedio se logró con un camino muy accidentado, no con una trayectoria estable. Esto ya adelanta una conclusión importante: la hipótesis de la Parte 1 ("volatilidad moderada pero sensible a choques externos") se confirma solo parcialmente — la sensibilidad a choques externos sí se confirma con claridad, pero la volatilidad medida no es moderada, sino alta incluso para un banco regional de mediana capitalización.

## 7. Identificación de periodos relevantes

Se identifican 7 periodos (más de los 5 mínimos solicitados) a partir de los quiebres de tendencia observados en la serie de precios, cada uno con un evento identificable en la sección 3 del reporte HTML.

In [1]:
periodos = [
    ("P1", "2018-01-01", "2019-04-30", "Incertidumbre política y cancelación del NAIM"),
    ("P2", "2020-02-20", "2020-09-10", "Crash por COVID-19 hasta el mínimo histórico"),
    ("P3", "2020-09-10", "2023-01-26", "Recuperación y ciclo alcista de tasas (con corrección intra-2022)"),
    ("P4", "2023-01-26", "2023-12-29", "Corrección tras el máximo histórico"),
    ("P5", "2024-06-03", "2024-06-04", "Choque electoral (jornada del 3-jun-2024)"),
    ("P6", "2024-06-04", "2025-03-31", "Aranceles de EE. UU. y normalización del margen"),
    ("P7", "2025-03-31", "2026-09-09", "Recuperación y transición de director general"),
]
filas=[]
for clave,ini,fin,desc in periodos:
    p_ini = precio_diario.loc[ini]; p_fin = precio_diario.loc[fin]
    var = (p_fin/p_ini - 1)*100
    filas.append([clave, ini, fin, f"${p_ini:.2f}", f"${p_fin:.2f}", f"{var:+.1f}%", desc])

tabla_periodos = pd.DataFrame(filas, columns=["Periodo","Inicio","Fin","Precio inicial","Precio final","Variación","Descripción"])
print(tabla_periodos.to_string(index=False))

Periodo     Inicio        Fin Precio inicial Precio final Variación                                                       Descripción
     P1 2018-01-01 2019-04-30         $33.47       $36.50     +9.1%                     Incertidumbre política y cancelación del NAIM
     P2 2020-02-20 2020-09-10         $30.21       $13.76    -54.4%                      Crash por COVID-19 hasta el mínimo histórico
     P3 2020-09-10 2023-01-26         $13.76       $77.68   +464.5% Recuperación y ciclo alcista de tasas (con corrección intra-2022)
     P4 2023-01-26 2023-12-29         $77.68       $57.33    -26.2%                               Corrección tras el máximo histórico
     P5 2024-06-03 2024-06-04         $63.20       $56.90    -10.0%                         Choque electoral (jornada del 3-jun-2024)
     P6 2024-06-04 2025-03-31         $56.90       $41.90    -26.4%                   Aranceles de EE. UU. y normalización del margen
     P7 2025-03-31 2026-09-09         $41.90       $57.67    +

**Nota sobre P1:** el signo del cambio de inicio a fin del periodo no contradice la idea de "incertidumbre": dentro de ese rango el precio cayó durante buena parte de 2018 (rendimiento anual 2018: -14.3%, sección 6) tras la cancelación del NAIM, y solo se recupera con fuerza en el primer cuatrimestre de 2019; se agrupa como un solo periodo narrativo de "ida y vuelta" político/macro previo a la pandemia, y se explica con detalle en el reporte HTML.

## 8. Máximo drawdown

Se calcula sobre la serie **diaria** completa como la caída porcentual máxima desde cualquier máximo acumulado (*running maximum*) hasta el valle posterior más bajo, antes de que el precio vuelva a superar ese máximo.

In [1]:
maximo_acumulado = precio_diario.cummax()
drawdown = precio_diario/maximo_acumulado - 1

fecha_valle = drawdown.idxmin()
dd_maximo = drawdown.min()
fecha_pico = precio_diario.loc[:fecha_valle].idxmax()
precio_pico = precio_diario.loc[fecha_pico]
precio_valle = precio_diario.loc[fecha_valle]

print("=== MÁXIMO DRAWDOWN (2017-2026) ===")
print(f"Pico previo:  {fecha_pico.date()}   Precio: ${precio_pico:.2f}")
print(f"Valle:        {fecha_valle.date()}   Precio: ${precio_valle:.2f}")
print(f"Caída:        {dd_maximo*100:.2f}%")
print(f"Duración pico->valle: {(fecha_valle-fecha_pico).days} días ({(fecha_valle-fecha_pico).days/30:.1f} meses)")

recuperado = precio_diario.loc[fecha_valle:][precio_diario.loc[fecha_valle:] >= precio_pico]
if len(recuperado):
    fecha_recup = recuperado.index[0]
    print(f"Recuperación del nivel previo: {fecha_recup.date()}  ({(fecha_recup-fecha_valle).days} días después del valle)")

dd_desde_ath = precio_diario.loc["2023-01-26":]/precio_diario.loc["2023-01-26"] - 1
print(f"\nPara referencia — caída máxima desde el techo de ene-2023 hasta el valle de 2025: {dd_desde_ath.min()*100:.1f}% "
      f"(el {dd_desde_ath.idxmin().date()})")

=== MÁXIMO DRAWDOWN (2017-2026) ===
Pico previo:  2019-05-31   Precio: $39.80
Valle:        2020-09-10   Precio: $13.76
Caída:        -65.43%
Duración pico->valle: 468 días (15.6 meses)
Recuperación del nivel previo: 2022-02-07  (515 días después del valle)

Para referencia — caída máxima desde el techo de ene-2023 hasta el valle de 2025: -46.1% (el 2025-03-31)


**Nota:** al construir la serie con puntos mensuales/quincenales interpolados en vez de cotizaciones diarias reales, es posible que el drawdown "verdadero" (con datos tick a tick) sea ligeramente distinto al reportado aquí; sin embargo, como el pico y el valle relevantes coinciden con fechas verificadas o muy cercanas a datos de mercado reales (mayo-2019 vía cotización OTC, y 10-sep-2020 como mínimo histórico exacto), la magnitud del drawdown máximo (de entre 60% y 65%) es razonablemente robusta al método de interpolación.

## 9. Crecimiento del PIB de México y comparación con el rendimiento anual

Cifras de variación porcentual real anual del PIB, tomadas de los comunicados del INEGI (cifras actualizadas/definitivas más recientes disponibles para cada año).

In [1]:
pib_mexico = {
    2017: 2.1,   # INEGI, cifra original
    2018: 2.1,   # INEGI
    2019: -0.3,  # INEGI
    2020: -8.2,  # INEGI, cifra mas citada (existen revisiones puntuales a -8.4/-8.5)
    2021: 6.1,   # INEGI, cifra actualizada (revision de la cifra original de 4.8%)
    2022: 3.9,   # INEGI, cifra actualizada (revision de la cifra original de 3.1%)
    2023: 3.2,   # INEGI, cifra definitiva
    2024: 1.2,   # INEGI, cifra ajustada (21-feb-2025)
    2025: 0.6,   # INEGI, estimacion oportuna anual (afectada por aranceles de EE.UU.)
}

comparacion = pd.DataFrame({
    "PIB México (% real anual)": pd.Series(pib_mexico),
    "Rendimiento BBAJIO (%)": pd.Series(rend_anual)
}).dropna()
print(comparacion.round(2).to_string())

correlacion = comparacion["PIB México (% real anual)"].corr(comparacion["Rendimiento BBAJIO (%)"])
print(f"\nCorrelación PIB vs. rendimiento anual BBAJIO (2017-2025): {correlacion:.2f}")

      PIB México (% real anual)  Rendimiento BBAJIO (%)
2017                       2.10                   13.49
2018                       2.10                  -14.34
2019                      -0.30                    8.91
2020                      -8.20                  -11.22
2021                       6.10                   31.40
2022                       3.90                   71.53
2023                       3.20                   -7.98
2024                       1.20                  -13.57
2025                       0.60                    5.26

Correlación PIB vs. rendimiento anual BBAJIO (2017-2025): 0.47


**Lectura de la correlación (positiva pero moderada):** el activo **no** es un espejo del ciclo económico. Hay años alineados (2020: PIB y precio caen juntos; 2021: ambos suben) pero también divergencias marcadas:
- **2019** (PIB -0.3%, acción +8.9%): la recompra activa de acciones que BanBajío ejecutó ese año (sección 4) es un factor idiosincrático que sostuvo el precio pese a la economía débil.
- **2022** (PIB +3.9%, acción +71.5%): el rendimiento de la acción está dominado por el ciclo de **tasas de interés** (Banxico subió la tasa de referencia de 4% a 10.5% ese año), que no es lo mismo que el ciclo de crecimiento — un banco puede beneficiarse de tasas altas aunque el PIB crezca a un ritmo moderado.
- **2023** (PIB +3.2%, fuerte; acción -8.0%): divergencia clara. El precio ya había anticipado el pico de rentabilidad en enero (máximo histórico) y corrigió el resto del año pese a que la economía seguía creciendo con fuerza.
- **2024** (PIB +1.2%, acción -13.6%): aquí ambos coinciden en desacelerar, pero el motor principal de la caída de la acción fue puntual y político (choque electoral del 3-jun), no la desaceleración gradual del PIB.

La implicación para el riesgo: **el ciclo de tasas de interés y los choques político-regulatorios explican mejor los movimientos de BBAJIO que el ciclo de crecimiento del PIB por sí solo**, aunque ambos ciclos suelen moverse juntos en el mediano plazo.

## 10. Exportación de tablas y series para el reporte HTML

Se guardan en disco las series y tablas usadas para construir las gráficas SVG incrustadas en el reporte final.

In [1]:
precio_mensual.round(2).to_csv("precio_mensual_final.csv")
tabla_anual.to_csv("rendimiento_anual_final.csv")
tabla_periodos.to_csv("tabla_periodos_final.csv", index=False)
comparacion.to_csv("pib_vs_rendimiento_final.csv")
print("Archivos exportados: precio_mensual_final.csv, rendimiento_anual_final.csv,")
print("                     tabla_periodos_final.csv, pib_vs_rendimiento_final.csv")

Archivos exportados: precio_mensual_final.csv, rendimiento_anual_final.csv,
                     tabla_periodos_final.csv, pib_vs_rendimiento_final.csv


## 11. Conclusión de los cálculos

- La serie reconstruida está anclada a 5 hechos verificados con fuente puntual y validada adicionalmente contra datos de recompras BMV (2019) y una cotización OTC real convertida con tipo de cambio histórico (2019-2024), lo que da confianza en la ubicación temporal y la magnitud aproximada de cada movimiento relevante.
- El **máximo drawdown histórico** ronda **-65%**, entre un pico local de mediados de 2019 y el mínimo del 10-sep-2020 (\$13.76) — no coincide con el día más comentado en prensa (9-mar-2020), sino que refleja la dinámica propia de la banca mexicana durante 2020: los programas de diferimiento de pagos (CNBV/ABM) retrasaron el reconocimiento contable del riesgo de crédito varios meses, por lo que la acción siguió cayendo después de que el índice general ya se estabilizaba.
- Se identifican **7 periodos** relevantes (más de los 5 solicitados), cada uno anclado a un evento verificable.
- La correlación con el PIB es positiva pero moderada (~0.47): el ciclo de tasas de Banxico y los choques político-regulatorios (elección 2024, aranceles 2025) son, en la práctica, mejores explicaciones puntuales de los movimientos de BBAJIO que el crecimiento del PIB por sí solo.
- Las gráficas que acompañan este análisis (precio 2017–2026 con periodos señalados, y rendimiento anual vs. PIB) se generan a partir de exactamente estos mismos datos y se incrustan directamente en el reporte HTML como SVG (sin depender de imágenes externas), siguiendo el mismo formato visual que la Parte 1.